# 05. Hyperparameter Tuning


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import sys

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models import MLP, GRU, TCN, MODEL_REGISTRY
from src.preprocessing import load_cached_features, get_grouped_splits, create_dataloader, compute_class_weights
from src.train import train_model, run_cross_validation, get_training_device
from src.evaluate import predict, evaluate_predictions

dev = get_training_device()
OUT = ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
print(f"Device: {dev}")

Device: cuda


## 1. 16f vs 32f Features

In [2]:
d16 = load_cached_features(ROOT / "data" / "processed" / "squat_pose_16f_v1.npz")
d32 = load_cached_features(ROOT / "data" / "processed" / "squat_pose_32f_v1.npz")

print(f"16-frame tensor shape: {d16['X'].shape} (clips, frames, features)")
print(f"32-frame tensor shape: {d32['X'].shape} (clips, frames, features)")

16-frame tensor shape: (150, 16, 51) (clips, frames, features)
32-frame tensor shape: (150, 32, 51) (clips, frames, features)


## 2. Cross-Validation Benchmark

In [3]:
print("Running 5-Fold CV on 16-frame data...")
cv_16 = run_cross_validation(ROOT / "data" / "processed" / "squat_pose_16f_v1.npz", n_folds=5, out_dir=OUT)

print("\nRunning 5-Fold CV on 32-frame data...")
cv_32 = run_cross_validation(ROOT / "data" / "processed" / "squat_pose_32f_v1.npz", n_folds=5, out_dir=OUT)

comparison = pd.DataFrame({
    "16-Frame Macro-F1": cv_16["Macro-F1"],
    "32-Frame Macro-F1": cv_32["Macro-F1"],
    "16-Frame Bal Acc":  cv_16["Bal Acc"],
    "32-Frame Bal Acc":  cv_32["Bal Acc"],
})
display(comparison)

Running 5-Fold CV on 16-frame data...
Starting 5-Fold Grouped CV on 25 subjects (16 frames per clip)...

  Fold 1/5 | Test subjects: [np.str_('subject_005'), np.str_('subject_010'), np.str_('subject_015'), np.str_('subject_020'), np.str_('subject_026')]


  MLP   | Acc=0.467 | Macro-F1=0.426 | Bal-Acc=0.467 | val-F1=0.374


  GRU   | Acc=0.433 | Macro-F1=0.417 | Bal-Acc=0.433 | val-F1=0.541


  TCN   | Acc=0.500 | Macro-F1=0.486 | Bal-Acc=0.500 | val-F1=0.583

  Fold 2/5 | Test subjects: [np.str_('subject_004'), np.str_('subject_009'), np.str_('subject_014'), np.str_('subject_019'), np.str_('subject_024')]
  MLP   | Acc=0.467 | Macro-F1=0.444 | Bal-Acc=0.467 | val-F1=0.580


  GRU   | Acc=0.533 | Macro-F1=0.531 | Bal-Acc=0.533 | val-F1=0.624


  TCN   | Acc=0.633 | Macro-F1=0.630 | Bal-Acc=0.633 | val-F1=0.664

  Fold 3/5 | Test subjects: [np.str_('subject_003'), np.str_('subject_008'), np.str_('subject_013'), np.str_('subject_018'), np.str_('subject_023')]


... (21 outputs left out)

  MLP   | Acc=0.467 | Macro-F1=0.400 | Bal-Acc=0.467 | val-F1=0.417


  GRU   | Acc=0.533 | Macro-F1=0.525 | Bal-Acc=0.533 | val-F1=0.500


  TCN   | Acc=0.567 | Macro-F1=0.499 | Bal-Acc=0.567 | val-F1=0.625

[RESULTS] Grouped 5-Fold Cross-Validation Summary (32f):
               Accuracy       Macro-F1        Bal Acc
model                                                
TCN       0.560 ± 0.083  0.537 ± 0.084  0.560 ± 0.083
GRU       0.533 ± 0.024  0.493 ± 0.065  0.533 ± 0.024
MLP       0.500 ± 0.071  0.474 ± 0.096  0.500 ± 0.071
Majority  0.500 ± 0.000  0.333 ± 0.000  0.500 ± 0.000

Saved CV metrics to D:\DL\outputs/cv_summary_32f.csv


,16-Frame Macro-F1,32-Frame Macro-F1,16-Frame Bal Acc,32-Frame Bal Acc
model,,,,
GRU,0.478 ± 0.071,0.493 ± 0.065,0.507 ± 0.043,0.533 ± 0.024
MLP,0.438 ± 0.036,0.474 ± 0.096,0.453 ± 0.038,0.500 ± 0.071
Majority,0.333 ± 0.000,0.333 ± 0.000,0.500 ± 0.000,0.500 ± 0.000
TCN,0.565 ± 0.078,0.537 ± 0.084,0.580 ± 0.077,0.560 ± 0.083


## 3. Model Capacity & Learning Rate Tuning

In [4]:
X32, y32, groups32 = d32["X"], d32["y"], d32["groups"]
tr_idx, va_idx, te_idx = get_grouped_splits(X32, y32, groups32, seed=42)
tr_loader = create_dataloader(X32, y32, tr_idx, batch_size=32, shuffle=True)
va_loader = create_dataloader(X32, y32, va_idx, batch_size=32, shuffle=False)
te_loader = create_dataloader(X32, y32, te_idx, batch_size=32, shuffle=False)
wt = compute_class_weights(y32[tr_idx], device=dev)

tuning_rows = []
for hidden_dim in [64, 96, 128]:
    for lr in [1e-3, 5e-4]:
        torch.manual_seed(42)
        m = GRU(input_dim=51, hidden_dim=hidden_dim)
        best_m, val_f1 = train_model(m, tr_loader, va_loader, class_weights=wt, lr=lr, device=dev, verbose=False)
        y_t, y_p, _ = predict(best_m, te_loader, device=dev)
        acc = (y_t == y_p).mean()
        metrics = evaluate_predictions(y_t, y_p)
        tuning_rows.append({
            "Hidden Dim": hidden_dim,
            "Learning Rate": lr,
            "Val Macro-F1": round(val_f1, 4),
            "Test Accuracy": round(acc, 4),
            "Test Macro-F1": round(metrics["macro_f1"], 4),
        })

tuning_df = pd.DataFrame(tuning_rows).sort_values("Val Macro-F1", ascending=False)
display(tuning_df)

,Hidden Dim,Learning Rate,Val Macro-F1,Test Accuracy,Test Macro-F1
2,96,0.0010,0.5662,0.4667,0.4570
3,96,0.0005,0.5333,0.5000,0.4857
4,128,0.0010,0.5333,0.5333,0.5249
0,64,0.0010,0.4994,0.5333,0.5333
5,128,0.0005,0.4994,0.5000,0.4712
1,64,0.0005,0.4223,0.5333,0.4034


## 4. Export Model Checkpoint

In [5]:
best_config = tuning_df.iloc[0]
print(f"Champion configuration: Hidden={int(best_config['Hidden Dim'])}, LR={best_config['Learning Rate']}")

torch.manual_seed(42)
champion_model = GRU(input_dim=51, hidden_dim=int(best_config["Hidden Dim"]))
champion_model, _ = train_model(champion_model, tr_loader, va_loader, class_weights=wt, lr=best_config["Learning Rate"], device=dev)

ckpt_32f_path = OUT / "gru_squat_32f.pt"
torch.save({
    "model_name": "GRU",
    "state_dict": champion_model.state_dict(),
    "frames": 32,
    "hidden_dim": int(best_config["Hidden Dim"]),
    "labels": {"0": "bad", "1": "good"},
    "val_f1": best_config["Val Macro-F1"],
    "test_accuracy": best_config["Test Accuracy"],
}, ckpt_32f_path)
print(f"Saved champion checkpoint to {ckpt_32f_path}")

Champion configuration: Hidden=96, LR=0.001


  Early stop at epoch 26
Saved champion checkpoint to D:\DL\outputs\gru_squat_32f.pt
